# 🎯 Tarea final · El solucionador de Cifras

En el concurso *Cifras y Letras* se sacan **6 números** (entre 1–10, 25, 50, 75 y 100) y un **objetivo** de 3 cifras. Hay que llegar al objetivo, o lo más cerca posible, combinando los números con **+ − × ÷**.

**Reglas**
- Cada número se usa **como máximo una vez** (no hace falta usarlos todos).
- Todos los resultados intermedios deben ser **enteros positivos**.

**Tu tarea:** escribe `resolver(numeros, objetivo)`, que devuelva una tupla `(valor, pasos)`, donde `pasos` es una lista de tuplas `(a, operación, b, resultado)`:

```python
resolver([100, 75, 50, 25, 6, 3], 952)
# (952, [(100, "+", 6, 106), (75, "×", 3, 225), ...])
```

| Nivel | Requisito |
| --- | --- |
| ⭐ | Encuentra **una** solución exacta si existe |
| ⭐⭐ | Si no la hay, devuelve **la más cercana** |
| ⭐⭐⭐ | Devuelve la solución con **menos operaciones** |
| ⭐⭐⭐⭐ | Resuelve **100 partidas en menos de 30 segundos** |
| ⭐⭐⭐⭐⭐ | Escribe `alcanzables(numeros)`: cuántos objetivos del 101 al 999 se pueden conseguir |

Solo puedes usar lo visto en el curso. Prepárate para **explicar tu diseño** y para **adaptarlo a una regla nueva** el día de la entrega.

---
## Herramientas (no modificar)

`nueva_partida()` genera una partida al azar, `comprobar()` verifica que una solución cumple las reglas y `mostrar()` la imprime.

In [ ]:
import random
import time

PLACAS = [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 25, 50, 75, 100]


def nueva_partida(semilla=None):
    """Devuelve (numeros, objetivo) al azar, como en el concurso."""
    rng = random.Random(semilla)
    return rng.sample(PLACAS, 6), rng.randint(101, 999)


def comprobar(numeros, resultado):
    """Comprueba que resultado = (valor, pasos) respeta las reglas.

    Devuelve el valor si todo es correcto; si no, lanza ValueError explicando el motivo.
    Cada paso es una tupla (a, operacion, b, r), con operacion en "+", "-", "×", "÷".
    """
    if resultado is None:
        raise ValueError("resolver debe devolver una tupla (valor, pasos), no None")
    valor, pasos = resultado
    disponibles = list(numeros)
    for a, operacion, b, r in pasos:
        for x in (a, b):
            if x not in disponibles:
                raise ValueError(f"{x} no está disponible en el paso {a} {operacion} {b}")
            disponibles.remove(x)
        if operacion == "+":
            correcto = a + b
        elif operacion == "-":
            correcto = a - b
        elif operacion == "×":
            correcto = a * b
        elif operacion == "÷":
            if b == 0 or a % b != 0:
                raise ValueError(f"División no exacta: {a} ÷ {b}")
            correcto = a // b
        else:
            raise ValueError(f"Operación desconocida: {operacion!r}")
        if r != correcto:
            raise ValueError(f"{a} {operacion} {b} no es {r}")
        if r <= 0:
            raise ValueError(f"Resultado no positivo: {a} {operacion} {b} = {r}")
        disponibles.append(r)
    if pasos and pasos[-1][3] != valor:
        raise ValueError(f"El valor {valor} no coincide con el último paso ({pasos[-1][3]})")
    if not pasos and valor not in numeros:
        raise ValueError(f"Sin pasos, el valor {valor} debería ser uno de los números")
    return valor


def mostrar(numeros, objetivo, resultado):
    """Muestra la solución de forma legible."""
    valor, pasos = resultado
    print(f"Números: {numeros}   Objetivo: {objetivo}")
    for a, operacion, b, r in pasos:
        print(f"   {a} {operacion} {b} = {r}")
    estado = "¡exacto!" if valor == objetivo else f"a {abs(valor - objetivo)} del objetivo"
    print(f"Resultado: {valor} ({estado}, {len(pasos)} operaciones)")


mostrar([100, 75, 50, 25, 6, 3], 952, (952, [(100, "+", 6, 106), (75, "×", 3, 225),
                                             (225, "×", 106, 23850), (23850, "-", 50, 23800),
                                             (23800, "÷", 25, 952)]))

---
## Tu solución

In [ ]:
def resolver(numeros, objetivo):
    ...  # TODO


def alcanzables(numeros):
    ...  # TODO (nivel 5)

---
## Tests

Ejecuta cada nivel cuando creas que lo tienes.

### ⭐ Nivel 1 · Solución exacta

In [ ]:
for numeros, objetivo in [([100, 75, 50, 25, 6, 3], 952), ([75, 3, 8, 1, 6, 50], 712), ([2, 3, 4, 5, 6, 7], 999), ([75, 50, 2, 3, 8, 7], 812)]:
    resultado = resolver(numeros, objetivo)
    assert resultado is not None, "resolver debe devolver (valor, pasos)"
    valor = comprobar(numeros, resultado)
    assert valor == objetivo, f"{numeros} -> {objetivo}: has llegado a {valor}, pero existe solución exacta"
    mostrar(numeros, objetivo, resultado)
    print()
print("⭐ Nivel 1 superado 🎉")

### ⭐⭐ Nivel 2 · La más cercana

In [ ]:
for numeros, objetivo, mejor_distancia in [([6, 100, 6, 10, 8, 10], 917, 1), ([1, 1, 2, 2, 3, 3], 111, 30)]:
    resultado = resolver(numeros, objetivo)
    valor = comprobar(numeros, resultado)
    assert abs(valor - objetivo) == mejor_distancia, (
        f"{numeros} -> {objetivo}: te has quedado a {abs(valor - objetivo)}, se puede quedar a {mejor_distancia}")
    mostrar(numeros, objetivo, resultado)
    print()
print("⭐⭐ Nivel 2 superado 🎉")

### ⭐⭐⭐ Nivel 3 · Menos operaciones

In [ ]:
for numeros, objetivo, minimo in [([25, 50, 75, 100, 3, 6], 150, 1), ([10, 10, 9, 9, 1, 1], 810, 2), ([75, 50, 2, 3, 8, 7], 812, 3), ([1, 2, 3, 4, 5, 6], 720, 4)]:
    resultado = resolver(numeros, objetivo)
    valor = comprobar(numeros, resultado)
    assert valor == objetivo, f"{numeros} -> {objetivo}: no es exacta"
    assert len(resultado[1]) == minimo, (
        f"{numeros} -> {objetivo}: usas {len(resultado[1])} operaciones, se puede con {minimo}")
print("⭐⭐⭐ Nivel 3 superado 🎉")

### ⭐⭐⭐⭐ Nivel 4 · 100 partidas en menos de 30 segundos

In [ ]:
mejores = [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

inicio = time.perf_counter()
for semilla, mejor_distancia in enumerate(mejores, start=1):
    numeros, objetivo = nueva_partida(semilla)
    valor = comprobar(numeros, resolver(numeros, objetivo))
    assert abs(valor - objetivo) == mejor_distancia, f"Partida {semilla} {numeros} -> {objetivo}: no es la mejor"
segundos = time.perf_counter() - inicio

print(f"100 partidas en {segundos:.1f} s")
assert segundos < 30, "Demasiado lento: ¿qué combinaciones estás explorando varias veces?"
print("⭐⭐⭐⭐ Nivel 4 superado 🎉")

### ⭐⭐⭐⭐⭐ Nivel 5 · Objetivos alcanzables

In [ ]:
for numeros, esperado in [([1, 2, 3, 4, 5, 6], 441), ([100, 75, 50, 25, 6, 3], 831), ([1, 1, 2, 2, 3, 3], 0)]:
    obtenido = alcanzables(numeros)
    assert obtenido == esperado, f"alcanzables({numeros}) = {obtenido}, esperaba {esperado}"
    print(f"{numeros}: {obtenido} de 899 objetivos")
print("⭐⭐⭐⭐⭐ Nivel 5 superado 🎉")